# Notebook 07 — SPINK7-KLK5 Production Campaign (GPU)

**Target Hardware:** NVIDIA A100 (40/80 GB) or H100 via Google Colab Pro+

This notebook executes the full production simulation pipeline for the
SPINK7-KLK5 binding free energy project:

1. Energy minimization ($\leq 10{,}000$ steps)
2. NVT equilibration (500 ps at 310 K)
3. NPT equilibration (1 ns at 310 K, 1 atm)
4. Production MD (100 ns, NPT)
5. SMD campaign (50 replicates, 2 ns each)
6. Umbrella sampling campaign (50 windows, 10 ns each)

## Estimated GPU Time

| Stage | Aggregate time | A100 estimate |
|-------|---------------|---------------|
| Minimization + equilibration | ~1.5 ns | ~15 min |
| Production MD | 100 ns | 8–12 h |
| SMD (50 $\times$ 2 ns) | 100 ns | 8–12 h |
| Umbrella (50 $\times$ 10 ns) | 500 ns | 40–60 h |
| **Total** | | **60–85 GPU h** |

## Cell 1 — Environment Setup

Install dependencies and verify CUDA/OpenCL platform availability.

In [ ]:
# Install packages (Colab only — skip in local environment).
# !pip install openmm mdtraj numpy scipy matplotlib pdbfixer biopython

import openmm

n_platforms = openmm.Platform.getNumPlatforms()
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(n_platforms)]
print(f"OpenMM version: {openmm.__version__}")
print(f"Available platforms: {platforms}")

# Prefer CUDA; fall back to OpenCL, then CPU.
PLATFORM_NAME = None
for name in ("CUDA", "OpenCL", "CPU"):
    if name in platforms:
        PLATFORM_NAME = name
        break
print(f"Selected platform: {PLATFORM_NAME}")
assert PLATFORM_NAME is not None, "No usable OpenMM platform found."

## Cell 2 — Mount Drive and Configure Paths

Mount Google Drive to load the prepared system and save outputs.
Set `DRIVE_ROOT` to the project folder on your Drive.

In [ ]:
import sys
from pathlib import Path

# ----- Colab drive mount (uncomment when running on Colab) -----
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = Path("/content/drive/MyDrive/SPINK7_KLK5")

# ----- Local development paths -----
PROJECT_ROOT = Path.cwd().parent  # When running from notebooks/
DRIVE_ROOT = PROJECT_ROOT

# Ensure src imports work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PREPARED_DIR = DRIVE_ROOT / "data" / "topologies"
PREPARED_PDB = DRIVE_ROOT / "data" / "pdb" / "prepared"
OUTPUT_DIR = DRIVE_ROOT / "data" / "production"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prepared system dir: {PREPARED_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## Cell 3 — Load Prepared System

Deserialize the OpenMM system XML and load the solvated PDB topology
produced by `scripts/run_prep.py`.

In [ ]:
from openmm import XmlSerializer
from openmm.app import PDBFile

# Locate the prepared system files.
system_xml_files = sorted(PREPARED_DIR.glob("*_system.xml"))
solvated_pdb_files = sorted(PREPARED_PDB.glob("*_solvated.pdb"))

assert len(system_xml_files) >= 1, f"No system XML found in {PREPARED_DIR}"
assert len(solvated_pdb_files) >= 1, f"No solvated PDB found in {PREPARED_PDB}"

SYSTEM_XML = system_xml_files[0]
SOLVATED_PDB = solvated_pdb_files[0]

print(f"System XML: {SYSTEM_XML}")
print(f"Solvated PDB: {SOLVATED_PDB}")

system = XmlSerializer.deserialize(SYSTEM_XML.read_text())
pdb = PDBFile(str(SOLVATED_PDB))
print(f"Loaded system: {system.getNumParticles()} particles")

## Cell 4 — Energy Minimization

Steepest descent minimization until convergence $< 10$ kJ/mol/nm,
up to 10,000 steps.

In [ ]:
from openmm.app import Simulation
from openmm import LangevinMiddleIntegrator, unit

from src.config import MinimizationConfig, EquilibrationConfig
from src.simulate.minimizer import minimize_energy
from src.simulate.platform import select_platform

min_config = MinimizationConfig()
eq_config = EquilibrationConfig()

integrator = LangevinMiddleIntegrator(
    eq_config.temperature_k * unit.kelvin,
    eq_config.friction_per_ps / unit.picosecond,
    eq_config.timestep_ps * unit.picoseconds,
)

platform, properties = select_platform(PLATFORM_NAME)
simulation = Simulation(pdb.topology, system, integrator, platform, properties)
simulation.context.setPositions(pdb.positions)

min_result = minimize_energy(simulation, min_config)
print(f"Minimization: {min_result['initial_energy_kj_mol']:.1f} -> "
      f"{min_result['final_energy_kj_mol']:.1f} kJ/mol "
      f"({min_result['n_steps']} steps)")

## Cell 5 — NVT + NPT Equilibration

- **NVT** (500 ps): Heat system to 310 K with heavy-atom restraints
  ($k = 1000$ kJ/mol/nm$^2$).
- **NPT** (1 ns): Density equilibration under Monte Carlo barostat
  at 1 atm, with gradual restraint release.

Physical invariants enforced:
- **IV-2:** $\langle T \rangle = 310 \pm 5$ K
- **IV-3:** $\rho = 1.0 \pm 0.05$ g/cm$^3$

In [ ]:
from src.simulate.equilibrate import run_nvt, run_npt

eq_output = OUTPUT_DIR / "equilibration"
eq_output.mkdir(parents=True, exist_ok=True)

nvt_result = run_nvt(simulation, eq_config, eq_output)
print(f"NVT: T_avg = {nvt_result['average_temperature_k']:.1f} K "
      f"(std = {nvt_result['temperature_std_k']:.2f} K)")

npt_result = run_npt(simulation, eq_config, eq_output)
print(f"NPT: density = {npt_result['average_density_g_cm3']:.4f} g/cm^3, "
      f"T_avg = {npt_result['average_temperature_k']:.1f} K")

EQUILIBRATED_STATE = npt_result["final_state_path"]
TOPOLOGY_PDB = npt_result.get("topology_path")
print(f"Equilibrated state saved: {EQUILIBRATED_STATE}")

## Cell 6 — Production MD (100 ns)

Unrestrained NPT dynamics at 310 K, 1 atm.
- Trajectory saved as DCD every 10 ps.
- Energy timeseries CSV logged every 10 ps.
- Checkpoints every 100 ps for restartability.

Physical invariant enforced:
- **IV-5:** NVE energy drift $< 0.1$ kJ/mol/ns/atom (reference segment)

In [ ]:
from src.config import ProductionConfig
from src.simulate.production import run_production

prod_config = ProductionConfig()
prod_output = OUTPUT_DIR / "production"
prod_output.mkdir(parents=True, exist_ok=True)

prod_result = run_production(simulation, prod_config, prod_output)
print(f"Production MD complete: {prod_result['total_time_ns']:.1f} ns, "
      f"{prod_result['n_frames']} frames")
print(f"Trajectory: {prod_result['trajectory_path']}")
print(f"Energy CSV: {prod_result['energy_timeseries_path']}")

## Cell 7 — SMD Campaign (50 Replicates)

Constant-velocity pulling of SPINK7 away from KLK5 along the initial
COM-COM axis:
- $k = 1000$ kJ/mol/nm$^2$, $v = 0.001$ nm/ps
- Pull distance: 3.0 nm over 2 ns per replicate
- 50 independent replicates for Jarzynski averaging

Physical invariants enforced:
- **IV-7:** All work values $W > 0$ (pulling does positive work)
- **IV-10:** Work distribution is unimodal

In [ ]:
from src.config import SMDConfig
from src.simulate.smd import run_smd_campaign

smd_config = SMDConfig()
smd_output = OUTPUT_DIR / "smd"
smd_output.mkdir(parents=True, exist_ok=True)

# Define pull groups: all CA atoms in each chain.
import mdtraj as md

ref_traj = md.load(str(TOPOLOGY_PDB or SOLVATED_PDB))
chain_ids = sorted({a.residue.chain.index for a in ref_traj.topology.atoms
                    if a.residue.name not in ('HOH', 'WAT', 'NA', 'CL')})
assert len(chain_ids) >= 2, "Need at least 2 protein chains for SMD pulling."

pull_group_1 = ref_traj.topology.select(f"chainid {chain_ids[0]} and name CA")
pull_group_2 = ref_traj.topology.select(f"chainid {chain_ids[1]} and name CA")

smd_results = run_smd_campaign(
    equilibrated_state_path=EQUILIBRATED_STATE,
    system_xml_path=SYSTEM_XML,
    config=smd_config,
    pull_group_1=pull_group_1,
    pull_group_2=pull_group_2,
    output_dir=smd_output,
    topology_pdb_path=TOPOLOGY_PDB or str(SOLVATED_PDB),
    platform_name=PLATFORM_NAME,
)

total_works = [r["total_work_kj_mol"] for r in smd_results]
print(f"SMD campaign: {len(smd_results)} replicates completed.")
print(f"Work range: {min(total_works):.1f} – {max(total_works):.1f} kJ/mol")

## Cell 8 — Umbrella Sampling Campaign (50 Windows)

Harmonic restraint sampling along the COM-COM separation coordinate $\xi$:
- $\xi$ range: 1.5–4.0 nm, spacing 0.05 nm → 50 windows
- $k = 1000$ kJ/mol/nm$^2$ per window
- 10 ns production per window (after 200 ps biased equilibration)

Physical invariant enforced:
- **IV-8:** Adjacent histogram overlap $\geq 10\%$

In [ ]:
from src.config import UmbrellaConfig
from src.simulate.umbrella import run_umbrella_campaign

umbrella_config = UmbrellaConfig()
umbrella_output = OUTPUT_DIR / "umbrella"
umbrella_output.mkdir(parents=True, exist_ok=True)

umbrella_results = run_umbrella_campaign(
    equilibrated_state_path=EQUILIBRATED_STATE,
    system_xml_path=SYSTEM_XML,
    config=umbrella_config,
    pull_group_1=pull_group_1,
    pull_group_2=pull_group_2,
    output_dir=umbrella_output,
    topology_pdb_path=TOPOLOGY_PDB or str(SOLVATED_PDB),
    platform_name=PLATFORM_NAME,
)

print(f"Umbrella campaign: {len(umbrella_results)} windows completed.")

## Cell 9 — Save Results to Drive

Copy all output files to Google Drive for persistence across Colab sessions.
Results can be analyzed offline using **Notebook 08** (convergence analysis)
and **Notebook 09** (invariant validation).

In [ ]:
import shutil

# Summary of outputs.
print("=== Production Campaign Summary ===")
print(f"Production trajectory: {prod_result['trajectory_path']}")
print(f"SMD replicates: {len(smd_results)}")
print(f"Umbrella windows: {len(umbrella_results)}")
print()
print("Proceed to Notebook 08 (convergence analysis) and")
print("Notebook 09 (invariant validation) for post-processing.")

# Uncomment to copy to Drive (Colab only):
# DRIVE_OUTPUT = Path("/content/drive/MyDrive/SPINK7_KLK5/production_results")
# DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
# shutil.copytree(str(OUTPUT_DIR), str(DRIVE_OUTPUT), dirs_exist_ok=True)
# print(f"Results saved to {DRIVE_OUTPUT}")